# ZIP-Code Income Shifts in Travis County

**Scenario:** A redistricting analyst needs to understand how income
distributions shifted across Travis County ZIP codes between tax years.
The IRS Statistics of Income (SOI) publishes ZIP-code-level income data
annually. The BLS QCEW provides employment context.

siege_utilities wraps both sources with URL builders and parse normalizers
so the analyst can focus on the question, not the data plumbing.

## 1. IRS SOI: Building Download URLs

IRS publishes SOI data as fixed-format CSVs on their website. The URL
pattern changes by tax year and state. `IRSSOIFiles.url_for()` encodes
this pattern so you don't have to reverse-engineer it each year.

In [1]:
from siege_utilities.economic.irs.soi import IRSSOIFiles, DEFAULT_CACHE_DIR

soi = IRSSOIFiles()

# URLs for Texas ZIP-code income data, two tax years
url_2020 = soi.url_for(tax_year=2020, state_abbrev="TX")
url_2018 = soi.url_for(tax_year=2018, state_abbrev="TX")

print(f"Tax year 2020: {url_2020}")
print(f"Tax year 2018: {url_2018}")
print(f"\nCache directory: {DEFAULT_CACHE_DIR}")
print(f"\nPattern: url_for(tax_year, state_abbrev) builds the IRS download URL")

Tax year 2020: https://www.irs.gov/pub/irs-soi/2020zpallagitx.csv
Tax year 2018: https://www.irs.gov/pub/irs-soi/2018zpallagitx.csv

Cache directory: /Users/dheerajchand/.siege_utilities/cache/irs_soi

Pattern: url_for(tax_year, state_abbrev) builds the IRS download URL


## 2. Parse Normalization

IRS CSV files have inconsistent formatting: ZIPCODE may or may not be
zero-padded, STATEFIPS varies between string and integer. The SOI parser
normalizes these automatically so downstream joins work reliably.

The `parse()` method returns a pandas DataFrame with standardized columns.
Since downloading the full IRS file requires network access, we demonstrate
the URL construction pattern and cache behavior here.

In [2]:
# The download → parse workflow (requires network for actual data)
# soi.download(tax_year=2020, state_abbrev="TX") → downloads to cache
# df = soi.parse(cached_path) → returns normalized DataFrame

# Show what the cache directory would contain
from pathlib import Path

cache_dir = Path(DEFAULT_CACHE_DIR)
print(f"Cache location: {cache_dir}")
print(f"Cache exists: {cache_dir.exists()}")
if cache_dir.exists():
    cached = list(cache_dir.glob("*.csv"))
    print(f"Cached files: {len(cached)}")
    for f in cached[:5]:
        print(f"  {f.name}")
else:
    print("No cached data yet — first download populates this directory")

Cache location: /Users/dheerajchand/.siege_utilities/cache/irs_soi
Cache exists: True
Cached files: 0


## 3. BLS QCEW: Employment Context

Income shifts without employment context tell half the story. The BLS
Quarterly Census of Employment and Wages (QCEW) provides establishment
counts, employment, and wages. `QCEWFiles` handles the download,
caching, and parsing with state/NAICS filtering.

In [3]:
from siege_utilities.economic.bls.qcew import QCEWFiles
import inspect

qcew = QCEWFiles()

# The QCEW workflow:
# 1. download(year) → fetches annual data to cache
# 2. load(year, quarter, state_fips, naics_depth) → filtered DataFrame
print("QCEWFiles API:")
print(f"  download: {inspect.signature(qcew.download)}")
print(f"  load:     {inspect.signature(qcew.load)}")
print(f"  parse:    {inspect.signature(qcew.parse)}")
print(f"\nCache: {qcew.cache_dir}")
print(f"\nExample usage:")
print(f"  df = qcew.load(year=2020, quarter=1, state_fips='48', naics_depth=2)")
print(f"  → Returns DataFrame with Travis County employment by 2-digit NAICS sector")

QCEWFiles API:
  download: (year: 'int') -> 'Path'
  load:     (*, year: 'int', quarter: 'int', state_fips: 'Optional[str]' = None, naics_depth: 'int' = 2) -> 'pd.DataFrame'
  parse:    (csv_path: 'Path', *, year: 'int', quarter: 'int', state_fips: 'Optional[str]' = None, naics_depth: 'int' = 2) -> 'pd.DataFrame'

Cache: /Users/dheerajchand/.siege_utilities/cache/qcew

Example usage:
  df = qcew.load(year=2020, quarter=1, state_fips='48', naics_depth=2)
  → Returns DataFrame with Travis County employment by 2-digit NAICS sector


## Key Patterns

- **url_for()** handles the IRS URL convention changes across tax years
- **parse()** normalizes zero-padding and type inconsistencies automatically
- **Cache directory** avoids re-downloading large files during iterative analysis
- **load()** combines download + parse + filtering in one call
- Combine IRS income data with QCEW employment data for a complete economic picture
  of how a geography is changing